In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# 1. the full gpt-oss-20b vocabulary
#
# This used to read list_all_tokens.npy: 28,973 strings, a genuine but partial
# slice of the model at 14.5% of it. That subset was missing real direction
# tokens (' arriba', ' oben', ' unten', ' kiri') and, worse, was missing '上'
# while keeping 下/左/右 -- so UP was asymmetrically under-covered in CJK purely
# as an artifact of the filtering. Start from every token the model has.
#
# Two directories, because they are two different things (the same scheme
# grid_tokens.ipynb uses, and both caches are shared with it):
#
#   CACHE_DIR  vocab_all_tokens.npy and vocab_form_embeddings.npy. Both derive from the
#              tokenizer alone -- no lens, no trajectories -- so they are shared.
#   OUT_DIR    this notebook's own products.
#
# CACHE_DIR resolves to the first path that already exists, then falls back to something
# always writable, so the notebook runs off the workstation as well as on it.
import os
from pathlib import Path

from transformers import AutoTokenizer

MODEL_ID = "openai/gpt-oss-20b"

_CACHE_CANDIDATES = [
    os.environ.get("INTERP_CACHE_DIR"),
    "/media/alex/D/Uni/northeastern/data/jlens",
    "/Users/alexdimofte/workspace/northeastern/data/jlens",
]
CACHE_DIR = Path(
    next((p for p in _CACHE_CANDIDATES if p and os.path.isdir(p)), Path.home() / ".cache" / "interp-vocab")
)


def _repo_root(start=Path.cwd()):
    """Nearest ancestor holding pyproject.toml, so OUT_DIR does not follow the kernel cwd."""
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists():
            return d
    return start


# Anchored to the repo root on purpose: .gitignore's `data/**` is root-anchored, so a
# cwd-relative "data" run from notebooks/ would land in notebooks/data/ -- not ignored,
# and committed by accident.
OUT_DIR = Path(os.environ.get("INTERP_OUT_DIR") or _repo_root() / "data")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"CACHE_DIR = {CACHE_DIR}")
print(f"OUT_DIR   = {OUT_DIR}")

VOCAB_NPY = CACHE_DIR / "vocab_all_tokens.npy"

if VOCAB_NPY.exists():
    list_all_tokens = np.load(VOCAB_NPY, allow_pickle=True)
    print(f"loaded cached vocabulary {VOCAB_NPY}")
else:
    _vtok = AutoTokenizer.from_pretrained(MODEL_ID)
    list_all_tokens = np.array([_vtok.decode([i]) for i in range(len(_vtok))], dtype=object)
    np.save(VOCAB_NPY, list_all_tokens)

print(f"{len(list_all_tokens)} tokens, {len(set(map(str, list_all_tokens)))} unique strings")

In [ ]:
# 2. token -> word pieces, and the vocabulary index everything else is scored against
import re

SEP_RE = re.compile(r"""[_\-./\\:,;()\[\]{}<>"'`|!?*+=#@$%^&~\s]+""")
CAMEL_RE = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")


def strip_token(tok):
    """Surface form with whitespace/punctuation shaved off: '_LEFT' -> 'LEFT'."""
    return SEP_RE.sub(" ", tok).strip()


def parts_of(tok):
    """Word pieces to match on, lowercased: '.moveLeft' -> ['move', 'left', 'moveleft']."""
    out = []
    for piece in SEP_RE.split(tok.strip()):
        if piece:
            out.extend(p for p in CAMEL_RE.split(piece) if p)
    whole = SEP_RE.sub("", tok.strip())
    if whole:
        out.append(whole)
    return list(dict.fromkeys(p.lower() for p in out))


# VOCAB_FORMS: every token the model has, normalised the same way a seed is. A seed or
# anchor may only be a string in here. Anything else is a word we invented, and it can
# reach the vocabulary only by *inexact* match, which is where the documented false
# positives came from (the removed .62 tier: aristera 186, sinistra 110, descendre 99,
# herunter 99).
VOCAB_FORMS = {strip_token(str(t)).lower() for t in list_all_tokens} - {""}


def admissible(groups, kind):
    """Keep only the seeds/anchors the model has as tokens, and print what was dropped."""
    print(f"{kind}: admissible = some vocabulary token strips/lowercases to it")
    kept = {}
    for d, words in groups.items():
        kept[d] = [w for w in words if strip_token(w).lower() in VOCAB_FORMS]
        gone = [w for w in words if strip_token(w).lower() not in VOCAB_FORMS]
        print(f"  {d:6s} {len(kept[d]):2d}/{len(words):2d} kept    dropped: {gone}")
    return kept


print(f"{len(VOCAB_FORMS)} normalised forms over {len(list_all_tokens)} tokens\n")
parts_of("_LEFT"), parts_of(" moveRight"), parts_of("/right")

In [ ]:
# 3. seeds
#
# THE RULE: every seed and every anchor must itself be a token in the model's vocabulary.
# The candidate set was always vocabulary-only; this extends that to the strings we score
# it against, so nothing outside the model decides what counts as a direction. The lists
# below are kept whole and filtered by admissible() at the bottom rather than pruned by
# hand, so the price of the rule stays visible in the output.
#
# LEX_SEEDS feeds difflib, so it is deliberately conservative: every short or
# English-homographic form is left out. Including them cost 7k false positives
# ('le', 'alt', 'sol', 'sus', 'len', 'opp', 'bal', 'der', 'port' all fire on
# ordinary code tokens; 'linke'/'linker'/'turun'/'onder'/'runter' pull in
# linked, line, turn, wonder, router at ratio .83-.91).
#
# 'monter' (French "to go up") is out for the same reason, and it is the one seed the
# guard change actually cost. It is a perfectly good token, but 16 of its 17 lexical
# hits were Monster / Monsters / Monte / Monterey / monaster / montr / montrer, and no
# cutoff removes them all -- the dictionary used to suppress them for free. The UP sense
# it carried is already covered by subir, sopra, arriba, boven, acima.
LEX_SEEDS = {
    "up": ["up", "upward", "upwards", "arriba", "subir", "dessus",
           "oben", "aufwaerts", "aufwärts", "sopra", "acima", "boven", "omhoog",
           "nahoru", "yukari", "yukarı", "epano", "πάνω", "вверх", "вгору", "ऊपर"],
    "down": ["down", "downward", "downwards", "abajo", "abaixo", "bajar",
             "dessous", "descendre", "unten", "herunter", "abwaerts", "abwärts",
             "beneden", "omlaag", "asagi", "aşağı", "κάτω", "вниз", "नीचे"],
    "left": ["left", "leftward", "leftwards", "izquierda", "izquierdo", "gauche",
             "sinistra", "sinistro", "esquerda", "esquerdo", "vanster", "vänster",
             "vasen", "doleva", "vlevo", "stanga", "stânga", "balra", "aristera",
             "αριστερά", "влево", "слева", "ліворуч", "बाएं"],
    "right": ["right", "rightward", "rightwards", "derecha", "derecho", "droite",
              "rechts", "rechte", "destra", "destro", "direita", "direito",
              "hoger", "höger", "oikea", "prawo", "prawa", "vpravo", "doprava",
              "dreapta", "jobbra", "deksia", "δεξιά", "вправо", "справа", "दाएं"],
}

# SEM_ANCHORS feeds the embedder, which scores meaning rather than characters, so
# ambiguous *spellings* are safe here. Two things are not.
#
# AMBIGUOUS MEANINGS. Auditing which anchor admitted which token caught these:
#   góra  -> Polish "up" AND "mountain": berg, hillside, Everest (12/12 junk)
#   prawo -> Polish "right" AND "law":   law, LAW, ley, legal (19)
#   links -> German "left" AND English "hyperlink": link, /link, (link
#   sol   -> Turkish "left" AND sun / musical note
#   bas   -> French "down" AND English bass: Bass, bassist, bask, bast
# The fix used to be a phrase that pins the directional sense ("en bas", "in giù",
# "al di sopra"). Phrases are no longer available -- no phrase is a single token -- so
# the fix is now to let the ambiguous form fall out of the anchor set entirely and take
# the recall loss. Two survivors have no phrase left to pin them and are the ones to
# watch in the cell-7 bands: 'prawa' (Polish right / genitive of law) and 'hoger'
# (Swedish right without the umlaut, but also Dutch "higher").
#
# HUB ANCHORS. Bare single syllables sit close to everything: '위' averaged .713
# cosine against the whole vocabulary, and with '아래' admitted 4,843 of 5,207
# semantic hits, mostly 1-3 char subword fragments. The two-syllable 위쪽 / 아래쪽
# ("upper side" / "lower side") that replaced them are not tokens either, so Korean
# drops out on both counts. What keeps the surviving hubs (上 下 左 右) safe is the
# per-anchor rank normalisation in cell 6, not the choice of anchor.
#
# Arrows go to all four directions: with only ← and → present, '↑' had no
# correct home and was landing under RIGHT. All four are tokens.
SEM_ANCHORS = {
    "up":    ["up", "upward", "above", "arriba", "en haut", "oben", "al di sopra",
              "em cima", "boven", "uppåt", "w górę", "вверх", "yukarı", "上",
              "위쪽", "di atas", "↑"],
    "down":  ["down", "downward", "below", "abajo", "en bas", "unten", "in giù",
              "em baixo", "beneden", "nedåt", "w dół", "вниз", "aşağı", "下",
              "아래쪽", "di bawah", "↓"],
    "left":  ["left", "izquierda", "gauche", "nach links", "a sinistra",
              "à esquerda", "vänster", "w lewo", "влево", "sol taraf", "左",
              "왼쪽", "ke kiri", "←"],
    "right": ["right", "derecha", "droite", "rechts", "a destra", "à direita",
              "höger", "w prawo", "вправо", "sağ taraf", "右", "오른쪽",
              "ke kanan", "→"],
}

LEX_SEEDS = admissible(LEX_SEEDS, "LEX_SEEDS")
SEM_ANCHORS = admissible(SEM_ANCHORS, "SEM_ANCHORS")

In [ ]:
# 4. lexical stage: difflib over the word pieces
#
# One guard is left: the ratio cutoff scales with seed length. .80 between 5-char
# strings just means "one char differs" (light/right, line/linke, turn/turun), so short
# seeds demand exact equality and everything else .85.
#
# The second guard is gone. It read /usr/share/dict/american-english and made an English
# word match at .95 instead of .85, on the grounds that a word already in the dictionary
# needs no direction to explain it. That word list was the last input to a score that did
# not come from the model, so it goes. What it used to suppress for free now shows up in
# the bands below -- counter/wonder/router against herunter, Wright/fright against right,
# Monster/Monterey against monter -- and the remedy is the one cell 3 already uses: drop
# a seed whose yield is mostly junk.
#
# There used to be a permissive .62 tier for long-vs-long pairs, to catch
# mangled foreign spellings. It does not survive the full vocabulary: at 29k
# tokens it cost 23 false positives, at 200k it cost 726, with four seeds
# generating two thirds of them (aristera 186, sinistra 110, descendre 99,
# herunter 99) and LEFT ending up 491 lexical-only hits out of 531. Those four seeds
# are gone anyway now -- none of them is a token.
from difflib import SequenceMatcher

flat_lex = [(w, d) for d, ws in LEX_SEEDS.items() for w in ws]


def min_ratio(part, seed):
    return 1.0 if len(seed) <= 4 else 0.85    # exact only for short seeds: 'up' scores .80 against 'cup'


_part_cache = {}


def score_part(part):
    hit = _part_cache.get(part)
    if hit is not None:
        return hit
    best, bw, bd = 0.0, "", ""
    lp = len(part)
    for w, d in flat_lex:
        floor = min_ratio(part, w)
        if 2 * min(lp, len(w)) / (lp + len(w)) < floor:
            continue              # cannot reach the cutoff whatever the overlap
        r = SequenceMatcher(None, part, w).ratio()
        if r >= floor and r > best:
            best, bw, bd = r, w, d
    _part_cache[part] = (best, bw, bd)
    return _part_cache[part]


lex_score, lex_seed, lex_dir = [], [], []
for n, t in enumerate(list_all_tokens):
    best, bw, bd = 0.0, "", ""
    for part in parts_of(str(t)):
        r, w, d = score_part(part)
        if r > best:
            best, bw, bd = r, w, d
    lex_score.append(best); lex_seed.append(bw); lex_dir.append(bd)
    if n % 50000 == 0:
        print(f"  {n:>6d}/{len(list_all_tokens)}  ({len(_part_cache)} parts cached)")
lex_score = np.array(lex_score)

print(f"\n{(lex_score > 0).sum()} tokens matched lexically")
for lo, hi in [(0.99, 1.01), (0.90, 0.99), (0.85, 0.90)]:
    idx = np.where((lex_score >= lo) & (lex_score < hi))[0]
    print(f"\n=== [{lo}, {hi}) : {len(idx)} ===")
    print("   ", [repr(str(list_all_tokens[i])) for i in idx[:40]])

In [ ]:
# 5. embedding model (CPU: the GTX 1050 is sm_61, unsupported by this torch build)
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
_tok = AutoTokenizer.from_pretrained(MODEL)
_mod = AutoModel.from_pretrained(MODEL).eval()


@torch.no_grad()
def encode(texts, batch=256, show_every=None):
    """Mean-pooled, L2-normalised embeddings. Bare strings, no template --
    'the direction "{x}"' compresses everything upward and kills the separation."""
    out = []
    for i in range(0, len(texts), batch):
        b = _tok(texts[i:i + batch], padding=True, truncation=True,
                 max_length=16, return_tensors="pt")
        h = _mod(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).float()
        out.append(F.normalize((h * m).sum(1) / m.sum(1), dim=-1))
        if show_every and (i // batch) % show_every == 0:
            print(f"  {i + len(b['input_ids']):>6d}/{len(texts)}")
    return torch.cat(out).numpy()

In [ ]:
# 6. semantic stage: centred embeddings, cosine against the anchors, rank-normalised
#
# The embedder scores similarity; it never supplies a word. Every anchor it is given
# came out of admissible() in cell 3, so the only strings in play are the model's own.
#
# Hubness bites on both sides of the comparison and each side needs its own fix.
#
# TOKEN SIDE -- the embedding space is anisotropic: every vector carries a large
# shared component, so short subword fragments come out similar to everything.
# Raw cosine >= .85 kept 38,679 of 150,233 forms, which is why no cosine
# threshold could ever work. Subtracting the mean form embedding removes that
# common direction and the top matches become almost purely genuine direction
# words in every script.
#
# ANCHOR SIDE -- anchors differ wildly in how close they sit to the vocabulary
# ('위' averaged .713, 'down' .357), so a max over anchors is a vote for the
# hubbiest one. Two normalisations were tried and each fixed only half:
#   z = (cos-mean)/std     damps hubs but leaves each anchor a different
#     ceiling. A perfect cos=1.0 scores +6.16 on 'down', +2.79 on 'right',
#     +2.08 on 'sağ' -- no RIGHT token could clear a global z=3.0 however
#     perfect, while junk like 'levantar' scored +3.99.
#   (cos-mean)/(1-mean)    equalises the ceiling at 1.0 but drops the damping,
#     and dividing by a small (1-mean) amplifies instead: '위' alone then
#     admitted 4,261 tokens, mostly 1-3 char fragments.
# Rank has both properties and assumes no distribution: a perfect match is the
# top percentile for every anchor whatever its hubbiness, and only the top k per
# anchor can be admitted. It is also what makes the surviving single-character
# anchors (上 下 左 右, and the arrows) safe to keep.
#
# "Which direction is it?" -> raw (centred) cosine, unnormalised. Normalising
# here over-corrects: '↑' matched anchor '↑' at cos=1.000 but only z=+2.88,
# versus '→' at cos=.725 yet z=+3.61, so a perfect match lost and '↑' was filed
# under RIGHT.
EMB_NPY = CACHE_DIR / "vocab_form_embeddings.npy"

anchor_words = [w for ws in SEM_ANCHORS.values() for w in ws]
anchor_dir = [d for d, ws in SEM_ANCHORS.items() for _ in ws]

# many tokens collapse to the same stripped form ('left', ' left', '_left', '.left')
uniq = sorted({strip_token(str(t)) for t in list_all_tokens} - {""})

# 150k forms is ~40 min on CPU, so the matrix is cached; delete the .npy to redo.
# The forms depend only on the vocabulary, not on the seeds, so grid_tokens.ipynb
# shares this exact file.
if EMB_NPY.exists():
    U = np.load(EMB_NPY)
    assert len(U) == len(uniq), f"cache is stale ({len(U)} vs {len(uniq)} forms), delete it"
    print(f"loaded cached embeddings {U.shape} from {EMB_NPY}")
else:
    U = encode(uniq, batch=256, show_every=100)
    np.save(EMB_NPY, U)

A = encode(anchor_words)


def unit(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)


centre = U.mean(0, keepdims=True)                       # the common component
S = unit(U - centre) @ unit(A - centre).T               # (n_uniq, n_anchors)
RANK = S.argsort(0).argsort(0) / (len(uniq) - 1)        # percentile per anchor

i_r = RANK.argmax(axis=1)                               # how strong -> threshold
i_c = S.argmax(axis=1)                                  # which way  -> label
by_form = {f: (float(RANK[i, i_r[i]]), anchor_words[i_c[i]], anchor_dir[i_c[i]])
           for i, f in enumerate(uniq)}

sem_rank, sem_seed, sem_dir = [], [], []
for t in list_all_tokens:
    r, w, d = by_form.get(strip_token(str(t)), (0.0, "", ""))
    sem_rank.append(r); sem_seed.append(w); sem_dir.append(d)
sem_rank = np.array(sem_rank)

print("\nsanity -- the literal direction words and the arrows:")
for a in ["up", "down", "left", "right", "↑", "↓", "←", "→"]:
    if a in by_form:
        s, w, d = by_form[a]
        print(f"  {a:6s} -> {d.upper():5s} (via {w}, rank={s:.5f})")
print(f"\nembedded {len(uniq)} unique forms for {len(list_all_tokens)} tokens")

In [ ]:
# 7. combine both signals into one frame
LEX_T = 0.01     # the guards in cell 4 already zeroed everything below bar
SEM_T = 0.9999   # percentile: top .01% of the vocabulary for some anchor.
                 # Inspecting the ranked list, forms 45-220 are near-uniformly
                 # genuine, 220-306 degrade, and past ~306 it is mostly code
                 # vocabulary (Font, Bash, DESCRIPTOR, Operator, syntax, Pdf).

df = pd.DataFrame({
    "token": [str(t) for t in list_all_tokens],
    "lex_score": lex_score, "lex_seed": lex_seed, "lex_dir": lex_dir,
    "sem_rank": sem_rank, "sem_seed": sem_seed, "sem_dir": sem_dir,
})
# cached so SEM_T can be retuned without recomputing anything
df.to_pickle(OUT_DIR / "direction_scores.pkl")

print("semantic bands (where to put SEM_T):")
for lo, hi in [(0.99995, 1.01), (0.9999, 0.99995), (0.9995, 0.9999),
               (0.999, 0.9995), (0.998, 0.999)]:
    sel = df[(df.sem_rank >= lo) & (df.sem_rank < hi)]
    print(f"  [{lo:.5f}, {hi:.5f}) : {len(sel):5d}  {[repr(t) for t in sel.token.head(12)]}")

print(f"\nlexical  : {(df.lex_score >= LEX_T).sum():5d} tokens")
print(f"semantic : {(df.sem_rank >= SEM_T).sum():5d} tokens")
print(f"union    : {((df.lex_score >= LEX_T) | (df.sem_rank >= SEM_T)).sum():5d} tokens")

In [ ]:
# 8. final candidate set
review = df[(df.lex_score >= LEX_T) | (df.sem_rank >= SEM_T)].copy()
review["hit"] = np.where(
    (review.lex_score >= LEX_T) & (review.sem_rank >= SEM_T), "both",
    np.where(review.lex_score >= LEX_T, "lexical", "semantic"),
)
review["direction"] = np.where(review.lex_score >= LEX_T, review.lex_dir, review.sem_dir)
review["rank"] = review[["lex_score", "sem_rank"]].max(axis=1)
review = review.sort_values(["direction", "hit", "rank"], ascending=[True, True, False])

print(f"{len(review)} candidates  "
      f"(both={sum(review.hit == 'both')}, "
      f"lexical only={sum(review.hit == 'lexical')}, "
      f"semantic only={sum(review.hit == 'semantic')})")
print(pd.crosstab(review.direction, review.hit).to_string())

# 'both' is essentially all true positives; the single-signal groups are where
# to spend review effort.
pd.set_option("display.max_rows", 800)
review[["token", "direction", "hit", "lex_score", "lex_seed", "sem_rank", "sem_seed"]]

In [ ]:
# 9. contamination probes
#
# The bands above were read by eye. These are the known-junk neighbours of the direction
# seeds -- every one of them is a word cell 4's comments name as a near miss -- so
# asserting they are absent turns that judgement call into something that fails loudly
# when a threshold moves. Keep reading the bands too: probes catch known failures, bands
# catch unknown ones. (grid_tokens.ipynb cell 9 does the same for the grid classes.)
PROBES = [" light", "Light", " bright", " Wright", " counter", " wonder", " router",
          " linked", " link", " links", " turn", " line", " port", " alt", " sol",
          " upload", " uploads", " downloads", " update", " upper", " opp", " len"]

kept = set(review.token)
leaked = [p for p in PROBES if p in kept]
print(f"{len(leaked)}/{len(PROBES)} contamination probes leaked into the candidate set")
for p in leaked:
    row = review[review.token == p].iloc[0]
    print(f"  {p!r:12s} -> {row.direction:5s} via {row.hit:8s} "
          f"lex={row.lex_score:.3f}({row.lex_seed!r}) sem={row.sem_rank:.5f}({row.sem_seed!r})")
if not leaked:
    print("  none")

In [ ]:
# 10. final output: {UP, DOWN, LEFT, RIGHT} -> raw, unprocessed tokens
#
# Everything above (stripping, lowercasing, camel splitting) was scoring
# machinery only. What lands here is the byte-exact vocabulary string --
# leading spaces, tabs, punctuation and all.
import json

direction_tokens = {
    d.upper(): review.loc[review.direction == d, "token"].drop_duplicates().tolist()
    for d in ["up", "down", "left", "right"]
}

originals = set(str(t) for t in list_all_tokens)
flat = [t for v in direction_tokens.values() for t in v]
for k, v in direction_tokens.items():
    assert all(t in originals for t in v), f"{k}: token was modified"
assert len(set(flat)) == len(flat), "token assigned to two directions"

print(f"{'':6s} {'n':>5s}   sample")
for k, v in direction_tokens.items():
    print(f"{k:6s} {len(v):5d}   {[repr(t) for t in v[:6]]}")

OUT = OUT_DIR / "direction_tokens_full.json"
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(direction_tokens, f, ensure_ascii=False, indent=2)

# round-trip check: the file must give back byte-identical strings
with open(OUT, encoding="utf-8") as f:
    assert json.load(f) == direction_tokens
print(f"\n{len(flat)} tokens -> {OUT}")